# Planar routes

The experiment where the flights are planar in 3D. 

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# %cd /content
# !rm -rf /content/CAN_Path_Integration_3D_model
# !git clone -b two-dimensional --single-branch https://github.com/jacobsvennevik/CAN_Path_Integration_Plane_estimation_3D_model.git /content/CAN_Path_Integration_3D_model
# !pip install -q git+https://github.com/jacobsvennevik/MADE.git
# %cd /content/CAN_Path_Integration_3D_model

# Local: this notebook lives in notebooks/experiments.
if not os.path.exists("config.py"):
    sys.path.insert(0, os.path.abspath("../.."))

from config import RunConfig, PlaneConfig, NetworkConfig, AnalysisConfig
from experiments.arena_3d_planar_routes import (
    Arena3DPlanarRoutesConfig, Arena3DPlanarRoutesExperiment,
    columnar_ground_truth,
)
from analysis.qy_fit import qy_fit, repeat_phase_misalignment
from analysis.rate_maps import occupancy_fraction
from analysis.structure3d import score_3d_from_map
from analysis.summaries import summarize_3d, summarize_qy

N_ROUTES = 25
ROUND_TRIPS = 2          # a later pass of each route
ENV_SIZE = 2.0
GRID_SPACING = 0.48
TARGET_SPEED = 0.004
SEED = 0

net = NetworkConfig(
    spacing=0.1,
    dim=2,
    lambda_net=1.28,
    ratio=1.05,
    alpha=1.806,
    b=1.0,
    offset_magnitude=0.196,
    dt=0.25,
    tau=5.0,
    velocity_gain=2.626,
    settle_steps=4000,
)

analysis = AnalysisConfig(field_stride=30)
RATEMAP_BINS = 20
N_SUB = 600
N_SHUFFLE = 0
SIGMA = AnalysisConfig.smooth_sigma
N_JOBS = max(1, (os.cpu_count() or 1) - 1)
G_VEC = np.array([0.0, 0.0, -9.81])

plane = PlaneConfig(
    kappa_v=300.0, kappa_sens=300.0,
    plane_mode="true",
    tau_refresh=1, seed=SEED,
)
exp_cfg = Arena3DPlanarRoutesConfig(
    env_size=ENV_SIZE, seed=SEED,
    n_routes=N_ROUTES, round_trips=ROUND_TRIPS, rest_steps=500,
    grid_spacing=GRID_SPACING, target_speed_rad_per_time=TARGET_SPEED,
    ratemap_bins=RATEMAP_BINS,
    ratemap_n_sub=N_SUB, ratemap_n_shuffle=N_SHUFFLE, ratemap_seed=SEED,
)
cfg = RunConfig(network=net, experiment=exp_cfg, plane=plane, analysis=analysis)
exp = Arena3DPlanarRoutesExperiment(cfg)


## Trajectory

The figure shows coverage. Occupancy in the title is the fraction of voxels visited.

In [ ]:
traj = exp.generate_trajectory(seed=SEED)
occ = occupancy_fraction(traj.world_pos, ENV_SIZE, RATEMAP_BINS, ndim=3)
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(traj.world_pos[:, 0], traj.world_pos[:, 1], lw=0.3)
ax.set_aspect("equal")
ax.set_title(f"seed {SEED}  occupancy {occ:.1%}")
plt.show()


## Both arms

The motion arm integrates the route normal. The columnar arm integrates the vertical normal on the same velocities. Both arms use the same cell seed. Each arm is scored in the room frame from its rate map, and with `qy_fit` on the strided field trace. The two fits use different spike seeds. `repeat_phase_misalignment` compares later flights of each route with the first. Outbound and return are separate routes.

In [ ]:
def run_arm(name, n_drive, torus_gt):
    print(f"\n=== {name} ===")
    res = exp.run(
        traj.world_pos, traj.v_body_seq, torus_gt, G_VEC,
        n_true_seq=n_drive, segment_starts=traj.segment_starts,
        plane=plane,
    )
    vol = score_3d_from_map(
        res.ratemap_sums, res.ratemap_counts,
        shuf_sums=res.shuf_sums, active_mask=res.active_mask,
        positions=res.field_pos, activity=res.field_act,
        grid_spacing=GRID_SPACING, env_size=ENV_SIZE,
        sigma=SIGMA, n_jobs=N_JOBS,
    )
    print(summarize_3d(vol, GRID_SPACING, ENV_SIZE))
    return res, vol


n_col, gt_col = columnar_ground_truth(traj, cfg.scale, net.dim)
res_motion, vol_motion = run_arm("motion", traj.n_true_seq, traj.torus_gt)
res_column, vol_column = run_arm("columnar", n_col, gt_col)
fitted = qy_fit(
    traj, res_motion.field_pos, res_motion.field_act, res_motion.field_stride,
    seed=SEED, arms=[(res_column.field_act, SEED + 1)],
)
lattice_motion, lattice_column = fitted
def repeat_rms(res):
    rows = repeat_phase_misalignment(res.theta_hist, traj.flight_table)
    rms = np.array([row["rms_rad"] for row in rows], dtype=float)
    med = float(np.median(rms)) if rms.size else float("nan")
    return rows, med

rep_m, rms_m = repeat_rms(res_motion)
rep_c, rms_c = repeat_rms(res_column)
arms = {
    "motion": dict(volume=vol_motion, lattice=lattice_motion, repeats=rep_m, repeat_rms=rms_m),
    "columnar": dict(volume=vol_column, lattice=lattice_column, repeats=rep_c, repeat_rms=rms_c),
}
print(summarize_qy(lattice_motion))
print(summarize_qy(lattice_column))


## Scores

Occupancy and median χ come from the volumetric rate map. The Qi and Yartsev columns are the median grid-fit score and the fraction of cells the fit calls significant. `repeat_rms` is the median, over routes, of the phase misalignment between repeats of one route, in radians. Outbound and return count separately.

In [ ]:
display(pd.DataFrame({
    name: {
        **summarize_3d(arm["volume"], GRID_SPACING, ENV_SIZE),
        **summarize_qy(arm["lattice"]),
        "repeat_rms": arm["repeat_rms"],
    }
    for name, arm in arms.items()
}).T)
